In [1]:
# Get the library files
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
dataset = pd.read_csv("DillibabuSarva_DefectDataset.csv")

In [3]:
# Display the dataset
dataset

,SHA,cbo,wmc,dit,rfc,lcom,totalMethods,totalFields,nosi,loc,...,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,variablesQty,maxNestedBlocks,uniqueWordsQty,defect
0,7a955fd6c7de2bd912be544dcfe77f9173a7aa600,5,60,2,55,189,27,5,30,247,...,4,2,47,9,27,5,17,3,191,0
1,000f1ab4780fc9460975791c52597f7c04e15be70,3,10,1,1,9,7,4,1,38,...,0,0,0,22,4,0,4,2,69,0
2,000f1ab4780fc9460975791c52597f7c04e15be71,3,10,1,1,9,7,4,0,38,...,0,0,0,22,4,0,4,2,69,1
3,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c270,20,59,3,63,189,24,9,4,262,...,0,6,6,14,45,8,41,4,222,0
4,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c271,21,58,2,61,189,24,9,0,260,...,0,6,6,14,45,8,41,4,222,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6047,ffd1ed788cbf10bed00d49d79c7ee44250c36ac11,52,124,12,144,963,110,9,0,804,...,0,0,26,16,32,4,30,6,689,1
6048,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c0,24,27,2,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,0
6049,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c1,22,27,1,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,1
6050,ffe7c9989a4553d35fd1d5041d0cece0a673a0c80,3,12,2,12,28,8,0,1,67,...,2,0,0,2,10,0,8,2,36,0


In [4]:
# Display the column names
dataset.columns

Index(['SHA', 'cbo', 'wmc', 'dit', 'rfc', 'lcom', 'totalMethods',
       'totalFields', 'nosi', 'loc', 'returnQty', 'loopQty', 'comparisonsQty',
       'tryCatchQty', 'parenthesizedExpsQty', 'stringLiteralsQty',
       'numbersQty', 'assignmentsQty', 'mathOperationsQty', 'variablesQty',
       'maxNestedBlocks', 'uniqueWordsQty', 'defect'],
      dtype='object')

In [5]:
# Feature selection result
selected_features = ['nosi','dit','cbo','rfc','maxNestedBlocks',
                     'uniqueWordsQty','assignmentsQty','numbersQty',
                     'tryCatchQty','parenthesizedExpsQty']

X = dataset[selected_features]
y = dataset['defect']

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
from sklearn.neural_network import MLPClassifier
classifier = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
#fitting the model for grid search
classifier.fit(X_train, y_train)

MLPClassifier(alpha=1e-05, hidden_layer_sizes=(5, 2), random_state=1,
              solver='lbfgs')

In [8]:
y_pred = classifier.predict(X_test)

In [9]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [10]:
print(cm)

[[908   0]
 [907   1]]


In [11]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [12]:
# NeuralNetMLP_Classification Report
print(clf_report)

              precision    recall  f1-score   support

           0       0.50      1.00      0.67       908
           1       1.00      0.00      0.00       908

    accuracy                           0.50      1816
   macro avg       0.75      0.50      0.33      1816
weighted avg       0.75      0.50      0.33      1816



In [13]:
# Finding the outliers for our dataset
Q1 = dataset[selected_features].quantile(0.25)
Q3 = dataset[selected_features].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
#Check
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 967
dit: Lesser = 0, Greater = 788
cbo: Lesser = 0, Greater = 403
rfc: Lesser = 0, Greater = 367
maxNestedBlocks: Lesser = 0, Greater = 411
uniqueWordsQty: Lesser = 0, Greater = 431
assignmentsQty: Lesser = 0, Greater = 490
numbersQty: Lesser = 0, Greater = 651
tryCatchQty: Lesser = 0, Greater = 651
parenthesizedExpsQty: Lesser = 0, Greater = 671


In [14]:
# Replacing the outliers with mean values
for col in selected_features:
    mode = dataset[col].mean()
    dataset[col] = np.where((dataset[col] > upper[col]) | (dataset[col] < lower[col]), mode, dataset[col])

In [15]:
# After replacing the outliers
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 0
dit: Lesser = 0, Greater = 0
cbo: Lesser = 0, Greater = 0
rfc: Lesser = 0, Greater = 0
maxNestedBlocks: Lesser = 0, Greater = 0
uniqueWordsQty: Lesser = 0, Greater = 0
assignmentsQty: Lesser = 0, Greater = 0
numbersQty: Lesser = 0, Greater = 0
tryCatchQty: Lesser = 0, Greater = 0
parenthesizedExpsQty: Lesser = 0, Greater = 0


In [16]:
A = dataset[selected_features]
b = dataset['defect']
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size = 0.3, random_state = 42, stratify = b)

In [17]:
classifier_recheck = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
classifier_recheck.fit(A_train, b_train)

MLPClassifier(alpha=1e-05, hidden_layer_sizes=(5, 2), random_state=1,
              solver='lbfgs')

In [18]:
b_pred = classifier_recheck.predict(A_test)

In [19]:
cmodel = confusion_matrix(b_test,b_pred)
print(cmodel)

[[  0 908]
 [  0 908]]


In [22]:
# NeuralNetMLPClassification Report after replacing outliers
import warnings
warnings.filterwarnings('ignore')
clf_report_check = classification_report(b_test, b_pred)
print(clf_report_check)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       908
           1       0.50      1.00      0.67       908

    accuracy                           0.50      1816
   macro avg       0.25      0.50      0.33      1816
weighted avg       0.25      0.50      0.33      1816



In [23]:
# Here, we could see accuracy accuracy same before and after the outliers removal